В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням алгоритмів бустингу: XGBoost та LightGBM, а також використаємо бібліотеку HyperOpt для оптимізації гіперпараметрів.

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Standard library
import sys
from typing import Tuple

# Third-party libraries
import numpy as np
import pandas as pd
import lightgbm as lgb
from xgboost import XGBClassifier
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Local application/library specific imports
sys.path.append('..')
from HW_2_3.process_bank_churn import preprocess_data, create_preprocessor, preprocess_new_data
from HW_2_3.custom_features import get_custom_pipelines
from HW_2_3.plot_utils import plot_roc_curve, print_main_metrics, plot_feature_importances

0. Зчитайте дані `train.csv` в змінну `raw_df` та скористайтесь наведеним кодом нижче аби розділити дані на трнувальні та валідаційні і розділити дані на ознаки з матириці Х та цільову змінну. Назви змінних `train_inputs, train_targets, train_inputs, train_targets` можна змінити на ті, які Вам зручно.

  Наведений скрипт - частина отриманого мною скрипта для обробки даних. Ми тут не викнуємо масштабування та обробку категоріальних змінних, бо хочемо це делегувати алгоритмам, які будемо використовувати. Якщо щось не розумієте в наведених скриптах, рекомендую розібратись: навичка читати код - важлива складова роботи в машинному навчанні.

In [4]:
def split_train_val(df: pd.DataFrame, target_col: str, test_size: float = 0.2, random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split the dataframe into training and validation sets.

    Args:
        df (pd.DataFrame): The raw dataframe.
        target_col (str): The target column for stratification.
        test_size (float): The proportion of the dataset to include in the validation split.
        random_state (int): Random state for reproducibility.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: Training and validation dataframes.
    """
    train_df, val_df = train_test_split(df, test_size=test_size, random_state=random_state, stratify=df[target_col])
    return train_df, val_df


def separate_inputs_targets(df: pd.DataFrame, input_cols: list, target_col: str) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Separate inputs and targets from the dataframe.

    Args:
        df (pd.DataFrame): The dataframe.
        input_cols (list): List of input columns.
        target_col (str): Target column.

    Returns:
        Tuple[pd.DataFrame, pd.Series]: DataFrame of inputs and Series of targets.
    """
    inputs = df[input_cols].copy()
    targets = df[target_col].copy()
    return inputs, targets

In [ ]:
raw_df = pd.read_csv('../assets/hw_2_2/train.csv')

drop_cols = ['id', 'CustomerId', 'Surname', 'Exited']
input_cols = [col for col in raw_df.columns if col not in drop_cols]
target_col = 'Exited'

train_df, val_df = split_train_val(raw_df, target_col=target_col)

train_inputs, train_targets = separate_inputs_targets(train_df, input_cols, target_col)
val_inputs, val_targets = separate_inputs_targets(val_df, input_cols, target_col)

print("Train shape:", train_inputs.shape, train_targets.shape)
print("Val shape:", val_inputs.shape, val_targets.shape)
raw_df.head()

1. В тренувальному та валідаційному наборі перетворіть категоріальні ознаки на тип `category`. Можна це зробити двома способами:
 1. `df[col_name].astype('category')`, як було продемонстровано в лекції
 2. використовуючи метод `pd.Categorical(df[col_name])`

In [ ]:
categorical_cols = ['Geography', 'Gender', 'NumOfProducts']

for col in categorical_cols:
    train_inputs[col] = pd.Categorical(train_inputs[col])
    val_inputs[col] = pd.Categorical(val_inputs[col])

print(train_inputs.dtypes[categorical_cols])

2. Навчіть на отриманих даних модель `XGBoostClassifier`. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів XGBoostClassifier - тут https://xgboost.readthedocs.io/en/stable/parameter.html#global-config

  **Важливо:** зробіть такі налаштування `XGBoostClassifier` аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Можна також, якщо працюєте в Google Colab, увімкнути можливість використання GPU (`Runtime -> Change runtime type -> T4 GPU`) і встановити параметр `device='cuda'` в `XGBoostClassifier` для пришвидшення тренування бустинг моделі.
  
  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням DecisionTrees раніше. Чи вийшло покращити якість?

In [ ]:
xgb_clf = XGBClassifier(
    max_depth=3,
    n_estimators=10,
    enable_categorical=True,
    missing=np.nan,
    device='cuda'
)

xgb_clf.fit(train_inputs, train_targets)

train_preds_proba = xgb_clf.predict_proba(train_inputs)[:, 1]
val_preds_proba = xgb_clf.predict_proba(val_inputs)[:, 1]

train_auc = roc_auc_score(train_targets, train_preds_proba)
val_auc = roc_auc_score(val_targets, val_preds_proba)

print(f"Train AUROC: {train_auc:.4f}")
print(f"Validation AUROC: {val_auc:.4f}")

plot_roc_curve(train_targets, train_preds_proba, title="ROC Curve - Train")
plot_roc_curve(val_targets, val_preds_proba, title="ROC Curve - Validation")

3. Використовуючи бібліотеку `Hyperopt` і приклад пошуку гіперпараметрів для `XGBoostClassifier` з лекції знайдіть оптимальні значення гіперпараметрів `XGBoostClassifier` для нашої задачі. Задайте свою сітку гіперпараметрів виходячи з тих параметрів, які ви б хотіли перебрати. Поставте кількість раундів в підборі гіперпараметрів рівну **20**.

  **Увага!** Для того, аби скористатись hyperopt, нам треба задати функцію `objective`. В ній ми маємо задати loss - це може будь-яка метрика, але бажано використовувтаи ту, яка цільова в вашій задачі. Чим менший лосс - тим ліпша модель на думку hyperopt. Тож, тут нам треба задати loss - негативне значення AUROC. В лекції ми натомість використовували Accuracy.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення гіперпараметрів
    - створіть в окремій зміній `final_clf` модель `XGBoostClassifier` з найкращими гіперпараметрами
    - навчіть модель `final_clf`
    - оцініть якість моделі `final_clf` на тренувальній і валідаційній вибірках з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пунктом (2) цього завдання?

In [ ]:
def objective(params):
    clf = XGBClassifier(
        n_estimators=int(params['n_estimators']),
        learning_rate=params['learning_rate'],
        max_depth=int(params['max_depth']),
        min_child_weight=params['min_child_weight'],
        subsample=params['subsample'],
        colsample_bytree=params['colsample_bytree'],
        gamma=params['gamma'],
        reg_alpha=params['reg_alpha'],
        reg_lambda=params['reg_lambda'],
        enable_categorical=True,
        missing=np.nan,
        device='cuda',
        random_state=42,
        tree_method='hist',
        early_stopping_rounds=10
    )

    clf.fit(
        train_inputs,
        train_targets,
        eval_set=[(val_inputs, val_targets)],
        verbose=False
    )

    val_preds = clf.predict_proba(val_inputs)[:, 1]
    val_auc = roc_auc_score(val_targets, val_preds)

    return {'loss': -val_auc, 'status': STATUS_OK}

space = {
    'n_estimators': hp.quniform('n_estimators', 50, 300, 25),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.3),
    'max_depth': hp.quniform('max_depth', 3, 12, 1),
    'min_child_weight': hp.quniform('min_child_weight', 1, 10, 1),
    'subsample': hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
    'gamma': hp.uniform('gamma', 0, 0.5),
    'reg_alpha': hp.uniform('reg_alpha', 0, 1),
    'reg_lambda': hp.uniform('reg_lambda', 0, 1)
}

trials = Trials()
best_xgb_params = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=20, trials=trials)

best_xgb_params['n_estimators'] = int(best_xgb_params['n_estimators'])
best_xgb_params['max_depth'] = int(best_xgb_params['max_depth'])
best_xgb_params['min_child_weight'] = int(best_xgb_params['min_child_weight'])

print("🔧 Best hyperparameters:", best_xgb_params)

final_clf = XGBClassifier(
    n_estimators=best_xgb_params['n_estimators'],
    learning_rate=best_xgb_params['learning_rate'],
    max_depth=best_xgb_params['max_depth'],
    min_child_weight=best_xgb_params['min_child_weight'],
    subsample=best_xgb_params['subsample'],
    colsample_bytree=best_xgb_params['colsample_bytree'],
    gamma=best_xgb_params['gamma'],
    reg_alpha=best_xgb_params['reg_alpha'],
    reg_lambda=best_xgb_params['reg_lambda'],
    enable_categorical=True,
    missing=np.nan,
    device='cuda',
    random_state=42,
    tree_method='hist'
)

final_clf.fit(train_inputs, train_targets)

train_preds = final_clf.predict_proba(train_inputs)[:, 1]
val_preds = final_clf.predict_proba(val_inputs)[:, 1]

train_auc = roc_auc_score(train_targets, train_preds)
val_auc = roc_auc_score(val_targets, val_preds)

print(f"✅ Train AUROC: {train_auc:.4f}")
print(f"✅ Validation AUROC: {val_auc:.4f}")

plot_roc_curve(train_targets, train_preds_proba, title="ROC Curve - Train")
plot_roc_curve(val_targets, val_preds_proba, title="ROC Curve - Validation")

4. Навчіть на наших даних модель LightGBM. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів LightGBM - тут https://lightgbm.readthedocs.io/en/latest/Parameters.html

  **Важливо:** зробіть такі налаштування LightGBM аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Аби передати категоріальні колонки в LightGBM - необхідно виявити їх індекси і передати в параметрі `cat_feature=cat_feature_indexes`

  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням XGBoostClassifier раніше. Чи вийшло покращити якість?

In [ ]:
cat_feature_indexes = [
    train_inputs.columns.get_loc(col)
    for col in train_inputs.select_dtypes(include="category").columns
]

lgb_clf = lgb.LGBMClassifier(
    max_depth=3,
    n_estimators=50,
    learning_rate=0.1,
    missing=np.nan,
    random_state=42
)

lgb_clf.fit(
    train_inputs,
    train_targets,
    eval_set=[(val_inputs, val_targets)],
    eval_metric="auc",
    categorical_feature=cat_feature_indexes,
)

train_preds_proba = lgb_clf.predict_proba(train_inputs)[:, 1]
val_preds_proba = lgb_clf.predict_proba(val_inputs)[:, 1]

train_auc = roc_auc_score(train_targets, train_preds_proba)
val_auc = roc_auc_score(val_targets, val_preds_proba)

print(f"Train AUROC: {train_auc:.4f}")
print(f"Validation AUROC: {val_auc:.4f}")

plot_roc_curve(train_targets, train_preds_proba, title="ROC Curve - Train")
plot_roc_curve(val_targets, val_preds_proba, title="ROC Curve - Validation")

5. Використовуючи бібліотеку `Hyperopt` і приклад пошуку гіперпараметрів для `LightGBM` з лекції знайдіть оптимальні значення гіперпараметрів `LightGBM` для нашої задачі. Задайте свою сітку гіперпараметрів виходячи з тих параметрів, які ви б хотіли перебрати. Поставте кількість раундів в підборі гіперпараметрів рівну **10**.

  **Увага!** Для того, аби скористатись hyperopt, нам треба задати функцію `objective`. І тут ми також ставимо loss - негативне значення AUROC, як і при пошуці гіперпараметрів для XGBoost. До речі, можна спробувати написати код так, аби в objective передавати лише модель і не писати схожий код двічі :)

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення гіперпараметрів
    - створіть в окремій зміній `final_lgb_clf` модель `LightGBM` з найкращими гіперпараметрами
    - навчіть модель `final_lgb_clf`
    - оцініть якість моделі `final_lgb_clf` на тренувальній і валідаційній вибірках з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пунктом (4) цього завдання?

In [ ]:
def objective(params):
    clf = lgb.LGBMClassifier(
        n_estimators=int(params['n_estimators']),
        learning_rate=params['learning_rate'],
        max_depth=int(params['max_depth']),
        num_leaves=int(params['num_leaves']),
        min_child_weight=params['min_child_weight'],
        subsample=params['subsample'],
        colsample_bytree=params['colsample_bytree'],
        reg_alpha=params['reg_alpha'],
        reg_lambda=params['reg_lambda'],
        min_split_gain=params['min_split_gain'],
        random_state=42
    )

    clf.fit(
        train_inputs,
        train_targets,
        eval_set=[(val_inputs, val_targets)],
        eval_metric='auc',
        categorical_feature=cat_feature_indexes,
        callbacks=[lgb.early_stopping(stopping_rounds=10)]
    )

    preds_proba = clf.predict_proba(val_inputs)[:, 1]
    auc = roc_auc_score(val_targets, preds_proba)

    return {'loss': -auc, 'status': STATUS_OK}

space = {
    'n_estimators': hp.quniform('n_estimators', 50, 300, 25),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
    'max_depth': hp.quniform('max_depth', 3, 12, 1),
    'num_leaves': hp.quniform('num_leaves', 20, 150, 1),
    'min_child_weight': hp.quniform('min_child_weight', 1, 10, 1),
    'subsample': hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
    'reg_alpha': hp.uniform('reg_alpha', 0.0, 1.0),
    'reg_lambda': hp.uniform('reg_lambda', 0.0, 1.0),
    'min_split_gain': hp.uniform('min_split_gain', 0.0, 0.1)
}

trials = Trials()
best_lgb_params = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=10, trials=trials)

best_lgb_params['n_estimators'] = int(best_lgb_params['n_estimators'])
best_lgb_params['max_depth'] = int(best_lgb_params['max_depth'])
best_lgb_params['num_leaves'] = int(best_lgb_params['num_leaves'])
best_lgb_params['min_child_weight'] = int(best_lgb_params['min_child_weight'])

print("Best hyperparameters:", best_lgb_params)

final_lgb_clf = lgb.LGBMClassifier(
    n_estimators=best_lgb_params['n_estimators'],
    learning_rate=best_lgb_params['learning_rate'],
    max_depth=best_lgb_params['max_depth'],
    num_leaves=best_lgb_params['num_leaves'],
    min_child_weight=best_lgb_params['min_child_weight'],
    subsample=best_lgb_params['subsample'],
    colsample_bytree=best_lgb_params['colsample_bytree'],
    reg_alpha=best_lgb_params['reg_alpha'],
    reg_lambda=best_lgb_params['reg_lambda'],
    min_split_gain=best_lgb_params['min_split_gain'],
    random_state=42
)

final_lgb_clf.fit(
    train_inputs,
    train_targets,
    eval_set=[(val_inputs, val_targets)],
    eval_metric='auc',
    categorical_feature=cat_feature_indexes
)

train_preds_proba = final_lgb_clf.predict_proba(train_inputs)[:, 1]
val_preds_proba = final_lgb_clf.predict_proba(val_inputs)[:, 1]

train_auc = roc_auc_score(train_targets, train_preds_proba)
val_auc = roc_auc_score(val_targets, val_preds_proba)

print(f"Train AUROC: {train_auc:.4f}")
print(f"Validation AUROC: {val_auc:.4f}")

plot_roc_curve(train_targets, train_preds_proba, title="ROC Curve - Train")
plot_roc_curve(val_targets, val_preds_proba, title="ROC Curve - Validation")

6. Оберіть модель з експериментів в цьому ДЗ і зробіть новий `submission` на Kaggle та додайте код для цього і скріншот скора на публічному лідерборді.
  
  **Напишіть коментар, чому ви обрали саме цю модель?**

  І я вас вітаю - це останнє завдання з цим набором даних 💪 На цьому етапі корисно проаналізувати, які моделі показали себе найкраще і подумати, чому.

In [ ]:
test_raw_df = pd.read_csv("../assets/hw_2_2/test.csv")
submission_df = pd.read_csv("../assets/hw_2_2/sample_submission.csv")

drop_cols = ["id", "CustomerId", "Surname", "Exited"]
input_cols_all = [col for col in raw_df.columns if col not in drop_cols]

X_all = raw_df[input_cols_all].copy()
y_all = raw_df["Exited"]

for col in X_all.select_dtypes(include='object').columns:
    X_all[col] = X_all[col].astype('category')
for col in test_raw_df.select_dtypes(include='object').columns:
    test_raw_df[col] = test_raw_df[col].astype('category')

X_test_processed = test_raw_df[input_cols_all].copy()

# ====== XGBoost Best Model ======
xgb_best = XGBClassifier(
    n_estimators=best_xgb_params['n_estimators'],
    learning_rate=best_xgb_params['learning_rate'],
    max_depth=best_xgb_params['max_depth'],
    min_child_weight=best_xgb_params['min_child_weight'],
    subsample=best_xgb_params['subsample'],
    colsample_bytree=best_xgb_params['colsample_bytree'],
    gamma=best_xgb_params['gamma'],
    reg_alpha=best_xgb_params['reg_alpha'],
    reg_lambda=best_xgb_params['reg_lambda'],
    enable_categorical=True,
    missing=np.nan,
    tree_method="hist",
    device="cuda",
    random_state=42
)
xgb_best.fit(X_all, y_all)
xgb_preds = xgb_best.predict(X_test_processed)
xgb_preds_proba = xgb_best.predict_proba(X_test_processed)[:, 1]

submission_xgb = submission_df.copy()
submission_xgb_proba = submission_df.copy()
submission_xgb["Exited"] = xgb_preds
submission_xgb_proba["Exited"] = xgb_preds_proba
submission_xgb.to_csv("../assets/hw_2_4/submission_xgb_hyperopt.csv", index=False)
submission_xgb_proba.to_csv("../assets/hw_2_4/submission_xgb_hyperopt_proba.csv", index=False)

# ====== LightGBM Best Model ======
lgb_best = lgb.LGBMClassifier(
    n_estimators=best_lgb_params['n_estimators'],
    learning_rate=best_lgb_params['learning_rate'],
    max_depth=best_lgb_params['max_depth'],
    num_leaves=best_lgb_params['num_leaves'],
    min_child_weight=best_lgb_params['min_child_weight'],
    subsample=best_lgb_params['subsample'],
    colsample_bytree=best_lgb_params['colsample_bytree'],
    reg_alpha=best_lgb_params['reg_alpha'],
    reg_lambda=best_lgb_params['reg_lambda'],
    min_split_gain=best_lgb_params['min_split_gain'],
    random_state=42
)
lgb_best.fit(X_all, y_all, categorical_feature=cat_feature_indexes)

lgb_preds = lgb_best.predict(X_test_processed)
lgb_preds_proba = lgb_best.predict_proba(X_test_processed)[:, 1]

submission_lgb = submission_df.copy()
submission_lgb_proba = submission_df.copy()
submission_lgb["Exited"] = lgb_preds
submission_lgb_proba["Exited"] = lgb_preds_proba
submission_lgb.to_csv("../assets/hw_2_4/submission_lgb_hyperopt.csv", index=False)
submission_lgb_proba.to_csv("../assets/hw_2_4/submission_lgb_hyperopt_proba.csv", index=False)

display(submission_xgb.head())
display(submission_xgb_proba.head())
display(submission_lgb.head())
display(submission_lgb_proba.head())

In [12]:
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

raw_df = pd.read_csv('../assets/hw_2_2/train.csv')

drop_cols = ['id', 'CustomerId', 'Exited']
input_cols = [col for col in raw_df.columns if col not in drop_cols]
target_col = 'Exited'

train_df, val_df = split_train_val(raw_df, target_col=target_col)

train_inputs, train_targets = separate_inputs_targets(train_df, input_cols, target_col)
val_inputs, val_targets = separate_inputs_targets(val_df, input_cols, target_col)

X = raw_df[input_cols].copy()
y = raw_df[target_col].copy()

cat_cols = train_inputs.select_dtypes(include=["object", "category"]).columns.tolist()

train_pool = Pool(data=train_inputs, label=train_targets, cat_features=cat_cols)
val_pool   = Pool(data=val_inputs,   label=val_targets,   cat_features=cat_cols)

def objective(params):
    params_casted = {
        "iterations": 9000,
        "learning_rate": float(params["learning_rate"]),
        "depth": int(params["depth"]),
        "l2_leaf_reg": float(params["l2_leaf_reg"]),
        "rsm": float(params["rsm"]),
        "bagging_temperature": float(params["bagging_temperature"]),
        "random_strength": float(params["random_strength"]),
        "min_data_in_leaf": int(params["min_data_in_leaf"]),
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "bootstrap_type": "Bayesian",
        "od_type": "Iter",
        "od_wait": int(params["od_wait"]),
        "random_seed": 3,
        "use_best_model": True,
        "task_type": "CPU",
        "verbose": False,
        "allow_writing_files": False
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []

    for tr_idx, va_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        train_pool = Pool(X_tr, y_tr, cat_features=cat_cols)
        val_pool   = Pool(X_va, y_va, cat_features=cat_cols)

        clf = CatBoostClassifier(**params_casted)
        clf.fit(train_pool, eval_set=val_pool)

        preds_proba = clf.predict_proba(val_pool)[:, 1]
        aucs.append(roc_auc_score(y_va, preds_proba))

    return {"loss": -float(np.mean(aucs)), "status": STATUS_OK}

space = {
    "learning_rate": hp.uniform("learning_rate", 0.01, 0.12),
    "depth": hp.quniform("depth", 4, 10, 1),
    "l2_leaf_reg": hp.loguniform("l2_leaf_reg", np.log(1e-2), np.log(1e2)),
    "rsm": hp.uniform("rsm", 0.5, 1.0),
    "bagging_temperature": hp.uniform("bagging_temperature", 0.0, 5.0),
    "random_strength": hp.loguniform("random_strength", np.log(1e-8), np.log(5.0)),
    "min_data_in_leaf": hp.quniform("min_data_in_leaf", 1, 128, 1),
    "od_wait": hp.quniform("od_wait", 300, 1200, 50),
}

trials = Trials()
best_cb_params = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=50,
    trials=trials,
    rstate=np.random.default_rng(3)
)

print(best_cb_params)


100%|██████████| 50/50 [1:34:30<00:00, 113.41s/trial, best loss: -0.9396079611341601]
{'bagging_temperature': 0.5213171872760074, 'depth': 4.0, 'l2_leaf_reg': 4.2861986065252395, 'learning_rate': 0.05652004079491241, 'min_data_in_leaf': 20.0, 'od_wait': 700.0, 'random_strength': 0.3192262765891149, 'rsm': 0.560533173632469}


In [13]:
test_raw_df = pd.read_csv("../assets/hw_2_2/test.csv")
submission_df = pd.read_csv("../assets/hw_2_2/sample_submission.csv")

input_cols_all = [col for col in raw_df.columns if col not in ["id", "CustomerId", "Exited"]]
X_all = raw_df[input_cols_all].copy()
y_all = raw_df["Exited"].copy()

cat_cols = X_all.select_dtypes(include=["object", "category"]).columns.tolist()
for c in cat_cols:
    X_all[c] = X_all[c].astype("category")
    test_raw_df[c] = test_raw_df[c].astype("category")

X_test_processed = test_raw_df[input_cols_all].copy()

cb_params_final = {
    "iterations": 9000,
    "learning_rate": float(best_cb_params["learning_rate"]),
    "depth": int(best_cb_params["depth"]),
    "l2_leaf_reg": float(best_cb_params["l2_leaf_reg"]),
    "rsm": float(best_cb_params["rsm"]),
    "bagging_temperature": float(best_cb_params["bagging_temperature"]),
    "random_strength": float(best_cb_params["random_strength"]),
    "min_data_in_leaf": int(best_cb_params["min_data_in_leaf"]),
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "bootstrap_type": "Bayesian",
    "use_best_model": False,
    "random_seed": 3,
    "task_type": "CPU",
    "verbose": False,
    "allow_writing_files": False
}

cb_best = CatBoostClassifier(**cb_params_final)

train_pool_all = Pool(X_all, y_all, cat_features=cat_cols)
cb_best.fit(train_pool_all)

cb_preds_proba = cb_best.predict_proba(X_test_processed)[:, 1]
cb_preds = (cb_preds_proba >= 0.5).astype(int)

submission_cb = submission_df.copy()
submission_cb_proba = submission_df.copy()
submission_cb["Exited"] = cb_preds
submission_cb_proba["Exited"] = cb_preds_proba

submission_cb.to_csv("../assets/hw_2_4/submission_cb_hyperopt_cross_val.csv", index=False)
submission_cb_proba.to_csv("../assets/hw_2_4/submission_cb_hyperopt_proba_cross_val.csv", index=False)

In [16]:
# GBC: base vs +poly (degree=2). Comments in English.
import numpy as np, pandas as pd, time
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.base import BaseEstimator, TransformerMixin

raw_df = pd.read_csv('../assets/hw_2_2/train.csv')
drop_cols = ['id','CustomerId','Exited']
y = raw_df['Exited'].copy()
X = raw_df[[c for c in raw_df.columns if c not in drop_cols]].copy()

cat_cols = X.select_dtypes(include=['object','category']).columns.tolist()
num_cols = X.columns.difference(cat_cols).tolist()

# split low/high-card; OHE only low-card
LOW_CARD_TH = 20
low_card = [c for c in cat_cols if X[c].nunique(dropna=False) <= LOW_CARD_TH]
# high-card example: Surname -> use TE later if нужно
if 'Surname' in low_card: low_card.remove('Surname')

# engineered features (keeps pandas DataFrame)
class FeatureMaker(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        Z = X.copy()
        if 'Age' in Z:
            Z['Age_log'] = np.log(np.clip(Z['Age'].astype(float), 1e-9, None))
        if {'Tenure','Age'}.issubset(Z.columns):
            a = np.log(np.clip(Z['Age'].astype(float), 1e-9, None))
            Z['TenureAgeRatio'] = Z['Tenure'].astype(float) / a
        if {'HasCrCard','IsActiveMember'}.issubset(Z.columns):
            Z['ActiveCrCard'] = Z['HasCrCard'].astype(float) + 4*Z['IsActiveMember'].astype(float) + 1
        return Z

# choose a SMALL poly set (don’t explode dims)
poly_cols = [c for c in ['CreditScore','Balance','EstimatedSalary','Tenure'] if c in X.columns]

# common pieces
numeric_base = ['Age','Balance','CreditScore','EstimatedSalary','Tenure','NumOfProducts',
                'Age_log','TenureAgeRatio','ActiveCrCard']
numeric_base = [c for c in numeric_base if c in (set(num_cols)|{'Age_log','TenureAgeRatio','ActiveCrCard'})]

num_tf_base = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
])

cat_tf_low = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# BASE pipeline
preprocess_base = ColumnTransformer(
    transformers=[
        ('num', num_tf_base, numeric_base),
        ('cat_low', cat_tf_low, low_card),
    ],
    remainder='drop'
)

pipe_base = Pipeline([
    ('feat', FeatureMaker()),
    ('prep', preprocess_base),
    ('clf', GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.08, max_depth=3,
        subsample=0.8, min_samples_leaf=20, max_features='sqrt',
        random_state=42
    ))
])

# +POLY(d2) on a small subset
num_without_poly = [c for c in numeric_base if c not in poly_cols]
preprocess_poly = ColumnTransformer(
    transformers=[
        ('num', num_tf_base, num_without_poly),
        ('poly', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('poly', PolynomialFeatures(degree=2, include_bias=False))
        ]), poly_cols),
        ('cat_low', cat_tf_low, low_card),
    ],
    remainder='drop'
)

pipe_poly = Pipeline([
    ('feat', FeatureMaker()),
    ('prep', preprocess_poly),
    ('clf', GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.08, max_depth=3,
        subsample=0.8, min_samples_leaf=20, max_features='sqrt',
        random_state=42
    ))
])

# run quick CV
def eval_pipe(name, pipe):
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    t0 = time.time()
    cv = cross_validate(pipe, X, y, cv=skf, scoring='roc_auc', n_jobs=-1, return_train_score=False)
    return {
        'setup': name,
        'mean_auc': cv['test_score'].mean(),
        'std_auc': cv['test_score'].std(),
        'fit_time_sum': cv['fit_time'].sum(),
        'wall_s': time.time()-t0
    }

res = [eval_pipe('BASE (log+ratios, OHE low-card)', pipe_base),
       eval_pipe('+POLY d2 on few nums', pipe_poly)]
print(pd.DataFrame(res).sort_values('mean_auc', ascending=False))


                             setup  mean_auc   std_auc  fit_time_sum    wall_s
0  BASE (log+ratios, OHE low-card)  0.934829  0.006140      3.135735  2.429239
1             +POLY d2 on few nums  0.934098  0.005451      4.589873  2.547923


In [17]:
# Paired CV comparison of two pipelines (Wilcoxon)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import wilcoxon
import numpy as np

def cv_scores(pipe, X, y, skf):
    scores = []
    for tr, va in skf.split(X, y):
        pipe.fit(X.iloc[tr], y.iloc[tr])
        p = pipe.predict_proba(X.iloc[va])[:,1]
        scores.append(roc_auc_score(y.iloc[va], p))
    return np.array(scores)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
s_base = cv_scores(pipe_base, X, y, skf)
s_poly = cv_scores(pipe_poly, X, y, skf)

print("BASE:", s_base.mean(), s_base)
print("POLY:", s_poly.mean(), s_poly)
print("Δ (base - poly):", (s_base - s_poly).mean())
print("Wilcoxon p-value:", wilcoxon(s_base, s_poly).pvalue)

BASE: 0.935675578939222 [0.92275293 0.93856163 0.9368681  0.93813229 0.94206295]
POLY: 0.9347167262481024 [0.92317374 0.938076   0.93379313 0.93674465 0.94179611]
Δ (base - poly): 0.0009588526911196382
Wilcoxon p-value: 0.1875
